In [1]:
from Tools import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import scipy.stats
import sys

# LaTex Formatierung in Plots
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern"
})

# pd.set_option("display.precision", 10)
# pd.set_option('display.float_format', lambda x: f'{x:.12e}')

c:\Users\oskar\OneDrive\UNI\AP1\GitHub\AP1-Gruppe-143\Tools.py:74: SyntaxWarning: invalid escape sequence '\c'
  new_str = f"$\cdot 10^{{{i}}}$"
c:\Users\oskar\OneDrive\UNI\AP1\GitHub\AP1-Gruppe-143\Tools.py:80: SyntaxWarning: invalid escape sequence '\c'
  new_str = "$\cdot 10$"
c:\Users\oskar\OneDrive\UNI\AP1\GitHub\AP1-Gruppe-143\Tools.py:83: SyntaxWarning: invalid escape sequence '\,'
  new_str = f"$\, \cdot 10^{{{i}}}$"
c:\Users\oskar\OneDrive\UNI\AP1\GitHub\AP1-Gruppe-143\Tools.py:289: SyntaxWarning: invalid escape sequence '\h'
  print(f"Da $z = \\left|\\frac{{\hat x-y}}{{\Delta x}}\\right| = \\left|\\frac{{ {Bestwert:.2f} {Einheit} - {Literaturwert:.2f} {Einheit}}}{{{Standardunsicherheit:.2f} {Einheit} }} \\right| = {z:.2f} > 2$, weicht unser Ergebnis Signifikant vom Literaturwert ab.")
c:\Users\oskar\OneDrive\UNI\AP1\GitHub\AP1-Gruppe-143\Tools.py:289: SyntaxWarning: invalid escape sequence '\D'
  print(f"Da $z = \\left|\\frac{{\hat x-y}}{{\Delta x}}\\right| = \\left|\\frac{{ 

In [2]:
# Formeln für Quadratisches bla

def a_hat(x, y):
    return (np.sum(x**2)*np.sum(y) - np.sum(x)*np.sum(x*y)) / (len(x) * np.sum(x**2) - (np.sum(x))**2)

def b_hat(x, y):
    return (len(x) * np.sum(x*y) - (np.sum(x) * np.sum(y)))/(len(x) * np.sum(x**2) - (np.sum(x))**2) 

def s_ (x, y):
    return np.sqrt(((1)/(len(x)-2)) * (np.sum(y - (a_hat(x, y) + b_hat(x, y) * x)))**2)

def a_delta (x, y):
    return s_(x, y) * np.sqrt( (np.sum(x**2)) / (len(x)*np.sum(x**2) - np.sum(x)**2))

def b_delta(x, y):
    return s_(x, y) * np.sqrt( len(x) / (len(x)*np.sum(x**2) - np.sum(x)**2))

def b_simpel(x, y):
    return (1 * np.sum(x*y)) / np.sum(x**2)

def Streuung_simpel(x, y):
    """Streuungswert s für Geraden ohne Achsenverschiebung."""
    return np.sqrt((1 / (len(x) - 1)) * np.sum((y - (b_hat(x, y) * x))**2))

def Streuung(x, y):
    """Streuungswert s für Geraden mit Achsenverschiebung."""
    return np.sqrt((1 / (len(x) - 2)) * np.sum((y - (a_hat(x, y) + b_hat(x, y) * x))**2))

def delta_b(x, s):
    return  s * np.sqrt(len(x) / (len(x) * np.sum(x*2) - np.sum(x)*2))


In [3]:
Schallgeschwindigkeit = 343.46 # m/s

# Auswertung Messung 1
messdaten = pd.read_excel("V23.xlsx", sheet_name="Messung1")
messdaten = messdaten.drop(columns=["Einstellungen"])

# SI Einheiten umrechnen
messdaten["Abstand"] = messdaten["Abstand"] * 0.01
messdaten["Frequenz"] = messdaten["Frequenz"] * 10e2

# Differenz zweier Abstände aka Wellenlänge
messdaten["diff"] = np.nan
for i in range(len(messdaten.index)-1):
    if i == len(messdaten.index)-1: continue
    else:
        messdaten.loc[i, "diff"] = messdaten["Abstand"][i+1] - messdaten["Abstand"][i]

# Literarisch Wellenlänge aus Frequenz
messdaten["LitWellenlänge"] = Schallgeschwindigkeit / messdaten["Frequenz"]
lambda_fehler = 4 * 10e-5 #m
freuquenz_fehler = 3 #Hz

# Wellenlänge Mittelwert
mittelwert_wellenlänge, mittelwert_fehler = gewichteter_Mittelwert(messdaten["diff"].dropna(), np.full(len(messdaten["diff"].dropna()), lambda_fehler))

# Geschwindigkeit
messdaten["Geschwindigkeit"] = messdaten["diff"] * messdaten["Frequenz"]
messdaten["Fehler Geschwindigkeit"] = np.sqrt((messdaten["Frequenz"].to_numpy()*lambda_fehler)**2 + (messdaten["diff"].to_numpy()*freuquenz_fehler)**2)

# Geschwindigkeit Mittelwert
mittelwert_geschwindigkeit, mittelwert_fehler_Geschwindigkeit = gewichteter_Mittelwert(messdaten["Geschwindigkeit"].dropna(), messdaten["Fehler Geschwindigkeit"].dropna())

print("Wellenlänge:", mittelwert_wellenlänge, "±", mittelwert_fehler)
print("Geschwindigkeit:", mittelwert_geschwindigkeit, "±", mittelwert_fehler_Geschwindigkeit)

# z-Test
print("z-Test für Schallgeschwindigkeit: ", zTest(mittelwert_geschwindigkeit, Literaturwert=Schallgeschwindigkeit, Standardunsicherheit=mittelwert_fehler_Geschwindigkeit))


if True: # Tabellen Printen
    cols = ['Frequenz', 'Abstand', 'diff', 'Geschwindigkeit', 'Fehler Geschwindigkeit']
    messdaten_tex = messdaten[cols]
    print(generate_Table(messdaten_tex, "Messreihe 1: Wellenlänge aus Phasenverschiebungen", "Messreihe1"))

# Plotten
if False:
    indexe = messdaten.index
    fig, ax = plt.subplots()#dpi=600)
    ax.errorbar(indexe, messdaten["diff"], yerr=lambda_fehler, fmt='.', capsize=3) #yerr=messdaten["Phasenverschiebung"].abs()*0.01,
    # ax.plot(indexe, messdaten["LitWellenlänge"], '.', color="red")#, markersize=2)
    ax.hlines(mittelwert_wellenlänge, xmin=min(indexe), xmax=max(indexe), color="red")
    ax.fill_between([min(indexe), max(indexe)], mittelwert_wellenlänge+mittelwert_fehler, mittelwert_wellenlänge-mittelwert_fehler, alpha=0.3, color="orange")

    ax.set_ylabel(r"$x_j-x_{j-1}$ (m)")
    ax.set_xlabel(r"$j$")
    ax.set_xticks(messdaten.index)
    ax.grid()
    plt.show()

Wellenlänge: 0.0085 ± 0.00013333333333333334
Geschwindigkeit: 344.89074176558887 ± 5.40988483426485
z-Test für Schallgeschwindigkeit:  0.26446806344692086
\begin{table}[H]
\centering
\caption{Messreihe 1: Wellenlänge aus Phasenverschiebungen}
\label{tab:Messreihe1}
\begin{tabular}{lllll}
\hline
Frequenz & Abstand & diff & Geschwindigkeit & Fehler Geschwindigkeit \\
\hline
4,04$\, \cdot 10^{4}$ & 6,17$\cdot 10^{-1}$ & 9,50$\cdot 10^{-3}$ & 3,84$\, \cdot 10^{2}$ & 1,62$\cdot 10$ \\
4,06$\, \cdot 10^{4}$ & 6,26$\cdot 10^{-1}$ & 7,70$\cdot 10^{-3}$ & 3,13$\, \cdot 10^{2}$ & 1,62$\cdot 10$ \\
4,05$\, \cdot 10^{4}$ & 6,34$\cdot 10^{-1}$ & 8,30$\cdot 10^{-3}$ & 3,36$\, \cdot 10^{2}$ & 1,62$\cdot 10$ \\
4,05$\, \cdot 10^{4}$ & 6,42$\cdot 10^{-1}$ & 8,40$\cdot 10^{-3}$ & 3,40$\, \cdot 10^{2}$ & 1,62$\cdot 10$ \\
4,06$\, \cdot 10^{4}$ & 6,51$\cdot 10^{-1}$ & 8,90$\cdot 10^{-3}$ & 3,61$\, \cdot 10^{2}$ & 1,62$\cdot 10$ \\
4,05$\, \cdot 10^{4}$ & 6,60$\cdot 10^{-1}$ & 8,00$\cdot 10^{-3}$ & 3,24$\,

In [4]:
# Auswertung Messung 2
messdaten = pd.read_excel("V23.xlsx", sheet_name="Messung2")
messdaten = messdaten.drop(columns=["Einstellungen"])

# SI Einheiten umrechnen
messdaten["Position"] = messdaten["Position"] * 0.01
messdaten["delta t"] = messdaten["delta t"] * 0.001

# Fehler
fehler_t = 0.03e-3 #s
fehler_position = 4 * 10e-5 #m

# lin Fit: 
linfit = scipy.stats.linregress(messdaten["Position"], messdaten["delta t"])
# Fehler der Steigung
m_f_Schallgeschwindigkeit = linfit.stderr / linfit.slope**2

print(f"=> Steigung: {linfit.slope:.4f} ± {linfit.stderr:.4f}, Achsenabschnitt: {linfit.intercept:.4f} ± {linfit.intercept_stderr:.4f}")
print(f"=> Schallgeschwindigkeit: {1/linfit.slope:.4f} ± {m_f_Schallgeschwindigkeit:.4f}")

# Plotten
if False:
    Residuendiagramm_manuell(messdaten["Position"], messdaten["delta t"], 
                            fehler_position, fehler_t, linfit.slope, linfit.intercept, linfit.stderr, linfit.intercept_stderr, 
                            x_label="Abstand (m)", y_label = r"$t$ (s)")


print(messdaten.columns)
if True: # Tabellen Printen
    # cols = x
    print(generate_Table(messdaten, "Messreihe 1: Wellenlänge aus Phasenverschiebungen", "Messreihe1"))

print("z-Test", zTest(1/linfit.slope, 343.46, m_f_Schallgeschwindigkeit))

=> Steigung: 0.0029 ± 0.0000, Achsenabschnitt: -0.0006 ± 0.0000
=> Schallgeschwindigkeit: 340.8960 ± 1.1864
Index(['Position', 'delta t'], dtype='str')
\begin{table}[H]
\centering
\caption{Messreihe 1: Wellenlänge aus Phasenverschiebungen}
\label{tab:Messreihe1}
\begin{tabular}{ll}
\hline
Position & delta t \\
\hline
7,18$\cdot 10^{-1}$ & 1,48$\cdot 10^{-3}$ \\
5,00$\cdot 10^{-1}$ & 8,30$\cdot 10^{-4}$ \\
6,00$\cdot 10^{-1}$ & 1,14$\cdot 10^{-3}$ \\
8,00$\cdot 10^{-1}$ & 1,73$\cdot 10^{-3}$ \\
9,00$\cdot 10^{-1}$ & 2,03$\cdot 10^{-3}$ \\
1,00 & 2,30$\cdot 10^{-3}$ \\
1,10 & 2,61$\cdot 10^{-3}$ \\
1,20 & 2,90$\cdot 10^{-3}$ \\
1,30 & 3,18$\cdot 10^{-3}$ \\
1,40 & 3,48$\cdot 10^{-3}$ \\
\hline
\end{tabular}
\end{table}

z-Test 2.1612387273179614
